# 完整 Workflow — 多節點串接與條件路由

**目標**:建立多節點 workflow、條件路由、使用 BaseWorkflow

**前置條件**:
- 已完成 [02_single_node_test.ipynb](02_single_node_test.ipynb)
- 理解節點的定義和測試


In [ ]:
from llm_framework.config import load_config
from llm_framework.llm_client import LLMClient
from llm_framework.workflow.base import BaseWorkflow
from llm_framework.workflow.state import WorkflowState, create_workflow_state
from llm_framework.mlflow.experiment import ExperimentManager

load_config("dev")


## Workflow 設計

流程:使用者輸入 → 意圖分類 → 條件路由 → 回答問題 或 執行指令


In [ ]:
def classify_intent(state: WorkflowState) -> dict:
    """分類使用者意圖"""
    client = LLMClient()
    user_input = state["metadata"].get("user_input", "")
    messages = [
        {"role": "system", "content": "判斷使用者意圖。回答 'question' 或 'command'。question=詢問資訊,command=要求執行動作。只回傳一個詞。"},
        {"role": "user", "content": user_input}
    ]
    response = client.chat(messages, temperature=0.0)
    intent = response.content.strip().lower()
    return {"metadata": {"intent": intent}}


## 路由邏輯


In [ ]:
def route_by_intent(state: WorkflowState) -> str:
    """根據意圖路由到不同節點"""
    intent = state["metadata"].get("intent", "question")
    if "command" in intent:
        return "execute_command"
    return "answer_question"


## 處理節點


In [ ]:
def answer_question(state: WorkflowState) -> dict:
    """回答問題"""
    client = LLMClient()
    user_input = state["metadata"].get("user_input", "")
    messages = [
        {"role": "system", "content": "你是知識助手,簡潔回答問題。"},
        {"role": "user", "content": user_input}
    ]
    response = client.chat(messages)
    return {"results": {"answer": response.content, "type": "question"}}

def execute_command(state: WorkflowState) -> dict:
    """執行指令"""
    client = LLMClient()
    user_input = state["metadata"].get("user_input", "")
    messages = [
        {"role": "system", "content": "你是執行助手,說明將如何執行指令。"},
        {"role": "user", "content": user_input}
    ]
    response = client.chat(messages)
    return {"results": {"action": response.content, "type": "command"}}


## 組裝 Workflow


In [ ]:
workflow = (
    BaseWorkflow("intent_router")
    .add_node("classify", classify_intent)
    .add_node("answer_question", answer_question)
    .add_node("execute_command", execute_command)
    .set_entry("classify")
    .add_conditional_edge("classify", route_by_intent, ["answer_question", "execute_command"])
    .set_finish(["answer_question", "execute_command"])
    .compile()
)

print("Workflow 已建立")


## 執行 Workflow


In [ ]:
manager = ExperimentManager()
with manager.start_run(run_name="test_question"):
    state = create_workflow_state(metadata={"user_input": "法國首都是哪裡?"})
    result = workflow.run(state)
    print(f"意圖: {result['metadata']['intent']}")
    print(f"類型: {result['results']['type']}")
    print(f"回應: {result['results'].get('answer', result['results'].get('action'))}")


## 測試不同輸入


In [ ]:
with manager.start_run(run_name="test_command"):
    state = create_workflow_state(metadata={"user_input": "幫我寄一封信給客戶"})
    result = workflow.run(state)
    print(f"意圖: {result['metadata']['intent']}")
    print(f"類型: {result['results']['type']}")
    print(f"回應: {result['results'].get('answer', result['results'].get('action'))}")


## 總結

你已經學會了:
- 建立多節點 Workflow
- 實作條件路由邏輯
- 使用 BaseWorkflow 組裝完整流程

**下一步**:[04_evaluation.ipynb](04_evaluation.ipynb) — 評估 LLM 品質
